# Meezan Branch Location Graph

Load the Meezan Excel file, create the branch hierarchy in Neo4j, and store branch coordinates as native Neo4j points.

## 1. Imports and configuration

In [18]:
from pathlib import Path
import os
import pandas as pd
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

PROJECT_ROOT = Path.cwd().parent
# xlsx_path = Path.cwd().parent / "data" / "raw" / "meezan.xlsx"
XLSX_PATH = PROJECT_ROOT / "data" / "raw" / "meezan.xlsx"

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
##
api_key = os.getenv("LLM_API_KEY")
model_name = os.getenv("LLM_MODEL", "gpt-4o-mini")

llm_kwargs = {"model": model_name, "api_key": api_key, "temperature": 0}
if os.getenv("LLM_BASE_URL"):
    llm_kwargs["base_url"] = os.getenv("LLM_BASE_URL")

llm = ChatOpenAI(**llm_kwargs)
##
DATASET_ID = "meezan_branches"
BATCH_SIZE = 250

# Create the connection once and reuse graph.query() in every later cell.
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

print("Neo4j connection ready.")
print("Excel file:", XLSX_PATH)
print("model:", model_name)

Neo4j connection ready.
Excel file: /home/ekkel-ai/momina/projects/graphrag/data/raw/meezan.xlsx
model: openai/gpt-oss-120b


## 2. Load the Excel file

In [21]:
df = pd.read_excel(
    XLSX_PATH,
    sheet_name="Sheet1",
    header=1,
    dtype=str,
)

df.columns = [str(column).strip() for column in df.columns]
df = df.apply(lambda column: column.str.strip() if column.dtype == "object" else column)

# Match the column names used in the Meezan workbook.
df = df.rename(columns={"City Name": "City", "Area Name": "Area"})

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Rows: 1102
Columns: ['S.No', 'Branch Code', 'Branch Name', 'OM Landline No', 'BM Landline No', 'Branch Landline Number', 'City', 'Area', 'Province', 'Branch Address', 'Latitude', 'Longitude']


,S.No,Branch Code,Branch Name,OM Landline No,BM Landline No,Branch Landline Number,City,Area,Province,Branch Address,Latitude,Longitude
0,1,105,"Marriott Hotel, Karachi",021-35693166,021-5682682,021-5683491,Karachi,Clifton,Sindh,"Marriott Hotel, Abdullah Haroon Road, Karachi",24.846098,67.0311
1,2,201,"Gulberg, Lahore",042-35755716\n(042-35777434),042-35870929,042-5879870-72,Lahore,Gulberg,Punjab,"60 Main Boulevard, Gulberg, Lahore",31.528028,74.350303
2,3,301,"Jinnah Avenue (Blue Area), Islamabad",051-2206513,051-2277518,051-2276712-15,Islamabad,Islamabad East,Islamabad,"Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ...",33.720747,73.076406
3,4,101,"PNSC (Lalazar), Karachi",021-35636253,021-35636252,021-35636404,Karachi,Chundrigar Road,Sindh,"37-A, Lalazar Area, Off Moulvi Tamizuddin Khan...",NaN,NaN
4,5,401,"Sereena Hotel, Faisalabad",041-2602594,041-2601225,041-2602595-97,Faisalabad,Commercial FSD,Punjab,"Shop No.G-737 to G-740, G-751 & G-753, Bazar C...",31.428856,73.082234


## 3. Prepare branch records

In [22]:
def clean_text(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value and value.lower() != "nan" else None


def clean_coordinate(value):
    value = clean_text(value)
    if value is None:
        return None
    try:
        return float(value)
    except ValueError:
        return None


required_columns = [
    "Branch Code", "Branch Name", "City", "Area", "Province",
    "Branch Address", "Latitude", "Longitude",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in Excel file: {missing_columns}")

records = []

for _, row in df.iterrows():
    branch_code = clean_text(row["Branch Code"])
    if not branch_code:
        continue

    # Meezan branch codes use four digits; preserve leading zeroes.
    branch_code = branch_code.split(".")[0].zfill(4)

    latitude = clean_coordinate(row["Latitude"])
    longitude = clean_coordinate(row["Longitude"])

    if latitude is not None and not -90 <= latitude <= 90:
        latitude = None
    if longitude is not None and not -180 <= longitude <= 180:
        longitude = None
    if latitude is None or longitude is None:
        latitude, longitude = None, None

    province = clean_text(row["Province"])
    city = clean_text(row["City"])
    area = clean_text(row["Area"])

    records.append({
        "dataset_id": DATASET_ID,
        "branch_code": branch_code,
        "branch_name": clean_text(row["Branch Name"]),
        "branch_address": clean_text(row["Branch Address"]),
        "city": city,
        "area": area,
        "province": province,
        "province_key": province.lower() if province else None,
        "city_key": f"{province.lower()}|{city.lower()}" if province and city else None,
        "area_key": f"{province.lower()}|{city.lower()}|{area.lower()}" if province and city and area else None,
        "latitude": latitude,
        "longitude": longitude,
    })

print("Branch records prepared:", len(records))
print("Records with coordinates:", sum(
    row["latitude"] is not None and row["longitude"] is not None
    for row in records
))

Branch records prepared: 1102
Records with coordinates: 525


## 4. Preview the prepared data

In [23]:
preview_df = pd.DataFrame(records)
display(preview_df.head(10))
print("Branches without coordinates:", preview_df["latitude"].isna().sum())

,dataset_id,branch_code,branch_name,branch_address,city,area,province,province_key,city_key,area_key,latitude,longitude
0,meezan_branches,0105,"Marriott Hotel, Karachi","Marriott Hotel, Abdullah Haroon Road, Karachi",Karachi,Clifton,Sindh,sindh,sindh|karachi,sindh|karachi|clifton,24.846098,67.031100
1,meezan_branches,0201,"Gulberg, Lahore","60 Main Boulevard, Gulberg, Lahore",Lahore,Gulberg,Punjab,punjab,punjab|lahore,punjab|lahore|gulberg,31.528028,74.350303
2,meezan_branches,0301,"Jinnah Avenue (Blue Area), Islamabad","Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ...",Islamabad,Islamabad East,Islamabad,islamabad,islamabad|islamabad,islamabad|islamabad|islamabad east,33.720747,73.076406
3,meezan_branches,0101,"PNSC (Lalazar), Karachi","37-A, Lalazar Area, Off Moulvi Tamizuddin Khan...",Karachi,Chundrigar Road,Sindh,sindh,sindh|karachi,sindh|karachi|chundrigar road,NaN,NaN
4,meezan_branches,0401,"Sereena Hotel, Faisalabad","Shop No.G-737 to G-740, G-751 & G-753, Bazar C...",Faisalabad,Commercial FSD,Punjab,punjab,punjab|faisalabad,punjab|faisalabad|commercial fsd,31.428856,73.082234
5,meezan_branches,0104,"Gulshan-e-Iqbal, Karachi","Shop No. 1, 2, 3 & 4, Plot No.B-41, Block 13-A...",Karachi,Gulshan-e-Iqbal,Sindh,sindh,sindh|karachi,sindh|karachi|gulshan-e-iqbal,24.906729,67.098657
6,meezan_branches,0402,"Kotwali Road, Faisalabad","P-66 Allama Iqbal Road (Kotwali Road), Faisalabad",Faisalabad,City Faisalabad,Punjab,punjab,punjab|faisalabad,punjab|faisalabad|city faisalabad,NaN,NaN
7,meezan_branches,0501,Nawan Shaher Branch Multan (Main Branch),"92,93,94, LMQ Road, Chowk Nawan Shaher, Multan",Multan,Ahmad Shah Abdali,Punjab,punjab,punjab|multan,punjab|multan|ahmad shah abdali,NaN,NaN
8,meezan_branches,0102,"Jodia Bazar, Karachi","H 9/A Darya Lal Street, Market Quarters Jodia ...",Karachi,Jodia Bazar,Sindh,sindh,sindh|karachi,sindh|karachi|jodia bazar,NaN,NaN
9,meezan_branches,0103,"F.T.C, Karachi","Ground Floor, Block-B, Finance & Trade Centre,...",Karachi,Shara-e-Faisal,Sindh,sindh,sindh|karachi,sindh|karachi|shara-e-faisal,24.859554,67.053578


Branches without coordinates: 577


## 5. Create constraints and indexes

These constraints keep branch codes unique within this dataset. Dataset-scoped keys prevent repeated city or area names in different provinces from being merged together.

In [25]:
schema_queries = [
    """CREATE CONSTRAINT branch_dataset_code_unique IF NOT EXISTS
       FOR (b:Branch) REQUIRE (b.dataset_id, b.branch_code) IS UNIQUE""",
    """CREATE CONSTRAINT province_dataset_key_unique IF NOT EXISTS
       FOR (p:Province) REQUIRE (p.dataset_id, p.name_key) IS UNIQUE""",
    """CREATE CONSTRAINT city_dataset_key_unique IF NOT EXISTS
       FOR (c:City) REQUIRE (c.dataset_id, c.name_key) IS UNIQUE""",
    """CREATE CONSTRAINT area_dataset_key_unique IF NOT EXISTS
       FOR (a:Area) REQUIRE (a.dataset_id, a.name_key) IS UNIQUE""",
]

for query in schema_queries:
    graph.query(query)

print("Constraints created")

Constraints created


## 6. Ingest branches into Neo4j

In [26]:
ingest_query = """
UNWIND $rows AS row

MERGE (p:Province {dataset_id: row.dataset_id, name_key: row.province_key})
SET p.name = row.province

MERGE (c:City {dataset_id: row.dataset_id, name_key: row.city_key})
SET c.name = row.city,
    c.province_key = row.province_key

MERGE (a:Area {dataset_id: row.dataset_id, name_key: row.area_key})
SET a.name = row.area,
    a.city_name = row.city,
    a.city_key = row.city_key,
    a.province_key = row.province_key

MERGE (b:Branch {dataset_id: row.dataset_id, branch_code: row.branch_code})
SET b.branch_name = row.branch_name,
    b.branch_address = row.branch_address,
    b.city = row.city,
    b.area = row.area,
    b.province = row.province,
    b.latitude = row.latitude,
    b.longitude = row.longitude,
    b.geocoding_status = CASE
        WHEN row.latitude IS NOT NULL AND row.longitude IS NOT NULL
        THEN 'coordinates_provided'
        ELSE 'coordinates_missing'
    END

MERGE (b)-[:LOCATED_IN]->(a)
MERGE (a)-[:WITHIN_CITY]->(c)
MERGE (c)-[:WITHIN_PROVINCE]->(p)
"""

for start in range(0, len(records), BATCH_SIZE):
    batch = records[start:start + BATCH_SIZE]
    graph.query(ingest_query, params={"rows": batch})
    print(f"Ingested {min(start + BATCH_SIZE, len(records))}/{len(records)}")

print("Branch graph ingestion complete.")

Ingested 250/1102
Ingested 500/1102
Ingested 750/1102
Ingested 1000/1102
Ingested 1102/1102
Branch graph ingestion complete.


## 7. Create native Neo4j spatial points

In [27]:
point_result = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
WHERE b.latitude IS NOT NULL AND b.longitude IS NOT NULL
SET b.location = point({latitude: b.latitude, longitude: b.longitude})
RETURN count(b) AS points_created
""", params={"dataset_id": DATASET_ID})

print("Spatial points created:", point_result[0]["points_created"])

graph.query("""
CREATE POINT INDEX branch_location_index IF NOT EXISTS
FOR (b:Branch)
ON (b.location)
""")
print("Point index created or already present.")

Spatial points created: 525
Point index created or already present.


## 8. Preview the graph

In [ ]:
counts = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
WITH count(b) AS branches
MATCH (a:Area {dataset_id: $dataset_id})
WITH branches, count(a) AS areas
MATCH (c:City {dataset_id: $dataset_id})
WITH branches, areas, count(c) AS cities
MATCH (p:Province {dataset_id: $dataset_id})
RETURN branches, areas, cities, count(p) AS provinces
""", params={"dataset_id": DATASET_ID})

display(pd.DataFrame(counts))

branch_preview = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
OPTIONAL MATCH (b)-[:LOCATED_IN]->(a:Area)
OPTIONAL MATCH (a)-[:WITHIN_CITY]->(c:City)
OPTIONAL MATCH (c)-[:WITHIN_PROVINCE]->(p:Province)
RETURN
    b.branch_code AS branch_code,
    b.branch_name AS branch_name,
    a.name AS area,
    c.name AS city,
    p.name AS province,
    b.latitude AS latitude,
    b.longitude AS longitude,
    b.location IS NOT NULL AS has_location
ORDER BY b.branch_code
LIMIT 15
""", params={"dataset_id": DATASET_ID})

display(pd.DataFrame(branch_preview))

## 9. Example: find nearby branches

In [3]:
# Replace these with the coordinates of the location to search from.
# search_latitude = 31.5204
# search_longitude = 74.3587
search_latitude = 31.51532
search_longitude = 74.35907

radius_m = 5000

nearby_branches = graph.query("""
MATCH (b:Branch {dataset_id: $dataset_id})
WHERE b.location IS NOT NULL
  AND point.distance(
      b.location,
      point({latitude: $latitude, longitude: $longitude})
  ) <= $radius_m
WITH b, point.distance(
    b.location,
    point({latitude: $latitude, longitude: $longitude})
) AS distance_m
RETURN
    b.branch_code AS branch_code,
    b.branch_name AS branch_name,
    b.city AS city,
    b.area AS area,
    round(distance_m / 1000.0, 2) AS distance_km
ORDER BY distance_m
LIMIT 10
""", params={
    "dataset_id": DATASET_ID,
    "latitude": search_latitude,
    "longitude": search_longitude,
    "radius_m": radius_m,
})

display(pd.DataFrame(nearby_branches))

,branch_code,branch_name,city,area,distance_km
0,0245,"Ghalib Market, Lahore",Lahore,Gulberg,0.54
1,1138,"Falcon Complex, Gulberg-III Lahore",Lahore,Gulberg,0.73
2,0290,"Liberty Market Branch, Lahore",Lahore,Gulberg,0.97
3,1160,"Tricon Corporate Center, Lahore",Lahore,Gulberg,1.51
4,0266,"Mini Market Gulberg II Branch, Lahore",Lahore,Gulberg,1.51
5,0216,"Cavalry Ground, Lahore",Lahore,Gulberg,1.58
6,0201,"Gulberg, Lahore",Lahore,Gulberg,1.64
7,0254,"Zahoor Elahi Road, Lahore (Ex: HSBC)",Lahore,Gulberg,1.67
8,0226,"Punjab Coopeartive Housing Society (PCHS), Lahore",Lahore,Defence Lahore,1.71
9,0294,"Jail Road Branch, Lahore\n(Ex: Zafar Ali Road ...",Lahore,Gulberg,2.63


In [14]:

import requests
import pandas as pd

# 1. Enter the location you want to search
search_location = "Jahangir Road, Lahore, Pakistan"

# 2. Geocode using your local Pakistan Nominatim server
NOMINATIM_URL = "http://localhost:8088/search"

response = requests.get(
    NOMINATIM_URL,
    params={
        "q": search_location,
        "format": "jsonv2",
        "limit": 1,
        "countrycodes": "pk",
        "addressdetails": 1,
    },
    headers={"User-Agent": "MeezanBranchSearch/1.0"},
    timeout=30,
)

response.raise_for_status()
results = response.json()

if not results:
    print("Location not found:", search_location)

else:
    result = results[0]

    latitude = float(result["lat"])
    longitude = float(result["lon"])

    print("Location:", result["display_name"])
    print("Latitude:", latitude)
    print("Longitude:", longitude)

    # 3. Find the five nearest Meezan branches in Neo4j
    nearest_branches = graph.query("""
        MATCH (b:Branch {dataset_id: $dataset_id})
        WHERE b.location IS NOT NULL

        WITH b,
             point.distance(
                 b.location,
                 point({
                     latitude: $latitude,
                     longitude: $longitude
                 })
             ) AS distance_m

        RETURN
            b.branch_code AS branch_code,
            b.branch_name AS branch_name,
            b.area AS area,
            b.city AS city,
            b.branch_address AS address,
            round(distance_m / 1000.0, 2) AS distance_km

        ORDER BY distance_m
        LIMIT 5
    """, params={
        "dataset_id": DATASET_ID,
        "latitude": latitude,
        "longitude": longitude,
    })

    # 4. Display results
    print("\nNearest Meezan branches:")
    display(pd.DataFrame(nearest_branches))


Location: Jahangir Road, ہربنس پُورہ, کوٹلی غازی, تحصیل شالیمار, ضلع لاہور, لاہور ڈویژن, پنجاب, 54850, پاکستان
Latitude: 31.5714965
Longitude: 74.4238846

Nearest Meezan branches:


,branch_code,branch_name,area,city,address,distance_km
0,0235,"Canal Bank Road Mughalpura, Lahore",Lahore Cantt,Lahore,"Plot # 125, St # 33, Naya Pul, Punj Pir Road, ...",1.28
1,0241,G.T Road Daroghawala,Business Lahore,Lahore,"Plot No.329-F, Main G.T Road, Daroghawala Lahore.",2.52
2,0244,"Mustafaabad Branch, Lahore",Gulberg,Lahore,"Meezan Bank Limited, Property # SE-6-R-148 All...",2.84
3,1150,Harbanspura,Lahore Cantt,Lahore,"Property No.116, Saeed Block, Canal Bank, Sala...",3.10
4,1146,Askari-X Lahore Cantt,Lahore Cantt,Lahore,"Shop No.6,7,14-15, Sector F, Askari X, Lahore ...",4.21


## Query Pipeline

### 1. get the graph schema

In [15]:
import json

# discover node labels and their properties
node_rows = graph.query("""
MATCH (n {dataset_id: $dataset_id})
UNWIND labels(n) AS label
UNWIND keys(n) AS property
RETURN DISTINCT label, property
ORDER BY label, property
""", params={"dataset_id": DATASET_ID})

# Discover relationship structure
relationship_rows = graph.query("""
MATCH (a {dataset_id: $dataset_id})-[r]->(b {dataset_id: $dataset_id})
RETURN DISTINCT
    labels(a) AS source,
    type(r) AS relationship,
    labels(b) AS target
ORDER BY relationship
""", params={"dataset_id": DATASET_ID})

schema = {
    "nodes": {},
    "relationships": relationship_rows
}

# Estimate property types from actual graph values
for row in node_rows:
    label = row["label"]
    prop = row["property"]
    schema["nodes"].setdefault(label, {"properties": {}})
    schema["nodes"][label]["properties"][prop] = "unknown"

for label, info in schema["nodes"].items():
    sample = graph.query(f"""
    MATCH (n:{label} {{dataset_id: $dataset_id}})
    RETURN properties(n) AS props
    LIMIT 20
    """, params={"dataset_id": DATASET_ID})

    for prop in info["properties"]:
        values = [
            row["props"].get(prop)
            for row in sample
            if row["props"].get(prop) is not None
        ]

        types = sorted({type(value).__name__ for value in values})
        info["properties"][prop] = ", ".join(types) if types else "unknown"

print("INPUT: Existing Neo4j graph")
print("\nOUTPUT: DISCOVERED SCHEMA")
print(json.dumps(schema, indent=2, ensure_ascii=False))

INPUT: Existing Neo4j graph

OUTPUT: DISCOVERED SCHEMA
{
  "nodes": {
    "Area": {
      "properties": {
        "city_key": "str",
        "city_name": "str",
        "dataset_id": "str",
        "name": "str",
        "name_key": "str",
        "province_key": "str"
      }
    },
    "Branch": {
      "properties": {
        "area": "str",
        "branch_address": "str",
        "branch_code": "str",
        "branch_name": "str",
        "city": "str",
        "dataset_id": "str",
        "geocoding_status": "str",
        "latitude": "float",
        "location": "WGS84Point",
        "longitude": "float",
        "province": "str"
      }
    },
    "City": {
      "properties": {
        "dataset_id": "str",
        "name": "str",
        "name_key": "str",
        "province_key": "str"
      }
    },
    "Province": {
      "properties": {
        "dataset_id": "str",
        "name": "str",
        "name_key": "str"
      }
    }
  },
  "relationships": [
    {
      "source": 

#### 2. LLM Planner

In [28]:
import json

def plan_query(user_question, schema):

    prompt = f"""
You are the query planner for a Meezan Bank branch search system.

DATABASE SCHEMA:
{json.dumps(schema, ensure_ascii=False)}

USER QUESTION:
{user_question}

Analyze the request and return valid JSON only:

{{
    "intent": "branch_search",
    "search_scope": "global",

    "location_clues": [],
    "location_query": None,
    "location_search_queries": [],
    "fallback_location_queries": [],

    "city": "Lahore",
    "province": None,

    "needs_geocoding": False,
    "needs_nearest": False,

    "branch_code": None,

    "filters": {{
        "city": "Lahore",
        "province": null,
        "area": null
    }},

    "requested_fields": [
        "branch_code",
        "branch_name",
        "branch_address",
        "city",
        "area"
    ],

    "ambiguities": []
}}

RULES:

1. Extract location clues exactly as expressed by the user.
2. Build location_query using the original location and explicitly provided city/province context.
3. Generate a small number of alternative location_search_queries.
   Only use the original clues and explicitly stated context.
   Do not invent nearby places or coordinates.
4. A location does not need to exist as a Branch, Area, or City
   in Neo4j. It may be an address or landmark that needs geocoding.
5. Set needs_geocoding=true when a location must be converted
   into coordinates for geographical search.
6. Set needs_nearest=true for nearest, closest, nearby, or proximity requests.
7. Determine search_scope carefully:
   - "local": User requests branches in a specific area,
     neighbourhood, road, or locality or mentioned a filter explicitly which match our db schema e.g branch code 3421.
   - "nearest": User explicitly asks for nearest, closest,
     nearby, or geographical proximity to a location.
   - "global": User requests all branches or branch listings
     within a city, province, or across the dataset.
8. A city or province alone should not trigger local search.
   For example:
   - "Branches in Lahore" -> global
   - "Branches in Punjab" -> global
   - "Branches in Gulberg" -> local
   - "Nearest branch to Gulberg, Lahore" -> nearest
9. Preserve city and province as filters when relevant.
   A nearest search can also have a city or province constraint.
10. Do not confuse a location filter with a geocoding target.
    Geocode only when geographical coordinates are needed
    for nearest-distance calculations.
11. Extract branch_code only when explicitly provided.
12. filters must contain only relevant filters supported by
    the supplied schema.
13. requested_fields must contain actual Branch properties
    from the supplied schema. Never invent properties.
14. If the location is unclear, preserve it and add an ambiguity.
15. Never invent coordinates, database records, or location matches.
16. Do not generate Cypher.
17. Return valid JSON only, without Markdown fences.

LOCATION FALLBACK RULES:

1. Preserve the exact user location in `location_query`.
2. Generate `location_search_queries` with progressively broader
   versions of the original location.
3. Generate `fallback_location_queries` containing potentially
   relevant nearby roads, neighbourhoods, or localities,
   when identifiable from the supplied context.
   for example, if the user asks for "Muskeenpura, Nabipura Road, Lahore then,",
  "fallback_location_queries": [
        "Nearby roads or localities around Nabipura Road, Lahore",
        "Jahangir Road, Lahore"
    ]
4. Never invent locations or claim proximity without verification.
5. Keep original and fallback queries separate.

"""

    response = llm.invoke(prompt)
    content = response.content.strip()

    if content.startswith("```"):
        content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    return json.loads(content)


# Test question
# user_question = "Tell me the nearest Meezan branch to Muskeenpura, Nabipura Road in Lahore"
# user_question = "Tell me all branches in lahore"
user_question = "Tell me details about the branch code 3601"

query_plan = plan_query(user_question, schema)

print("INPUT QUESTION:")
print(user_question)

print("\nOUTPUT: LLM SEARCH PLAN")
print(json.dumps(query_plan, indent=2, ensure_ascii=False))


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


INPUT QUESTION:
Tell me details about the branch code 3601

OUTPUT: LLM SEARCH PLAN
{
  "intent": "branch_search",
  "search_scope": "local",
  "location_clues": [],
  "location_query": null,
  "location_search_queries": [],
  "fallback_location_queries": [],
  "city": null,
  "province": null,
  "needs_geocoding": false,
  "needs_nearest": false,
  "branch_code": "3601",
  "filters": {
    "city": null,
    "province": null,
    "area": null
  },
  "requested_fields": [
    "branch_code",
    "branch_name",
    "branch_address",
    "city",
    "area"
  ],
  "ambiguities": []
}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


#### 3. Geocoding Helper Function

In [39]:
import requests

NOMINATIM_URL = "http://localhost:8088/search"

def geocode_location(plan):

    queries = plan.get("location_search_queries", [])

    # Always try the original location first
    location_query = plan.get("location_query")
    if location_query:
        queries = [location_query] + [
            q for q in queries if q != location_query
        ]

    for query in queries:

        print(f"Geocoding: {query}")

        response = requests.get(
            NOMINATIM_URL,
            params={
                "q": query,
                "format": "jsonv2",
                "limit": 1,
                "countrycodes": "pk",
                "addressdetails": 1
            },
            headers={"User-Agent": "MeezanBranchSearch/1.0"},
            timeout=30
        )

        response.raise_for_status()
        results = response.json()

        if results:
            result = results[0]

            location = {
                "latitude": float(result["lat"]),
                "longitude": float(result["lon"]),
                "matched_query": query,
                "display_name": result["display_name"]
            }

            print("Location found:", location["display_name"])
            return location

    raise ValueError(
        f"Could not geocode the requested location: {location_query}"
    )


#### 4. Search Functions

In [ ]:
import pandas as pd

# Common function to format branch results
def format_results(rows, requested_fields=None):
    if not rows:
        print("No matching branches found.")
        return

    df = pd.DataFrame(rows)

    if requested_fields:
        available = [field for field in requested_fields if field in df.columns]
        if available:
            df = df[available]

    display(df)


# 1. LOCAL SEARCH
# For branch code, area, or specific branch filters.

def local_search(plan):
    filters = plan.get("filters", {})

    branch_code = plan.get("branch_code")
    city = filters.get("city") or plan.get("city")
    province = filters.get("province") or plan.get("province")
    area = filters.get("area")
    location_clues = plan.get("location_clues", [])

    conditions = ["b.dataset_id = $dataset_id"]

    params = {
        "dataset_id": DATASET_ID,
        "branch_code": branch_code,
        "city": city,
        "province": province,
        "area": area,
        "location_clues": location_clues,
        "limit": plan.get("limit") or 10
    }

    if branch_code:
        conditions.append("b.branch_code = $branch_code")

    if city:
        conditions.append("toLower(b.city) = toLower($city)")

    if province:
        conditions.append("toLower(b.province) = toLower($province)")

    # Use location clues across multiple properties.
    if location_clues:
        conditions.append("""
            ALL(clue IN $location_clues WHERE
                toLower(coalesce(b.branch_name, '')) CONTAINS toLower(clue)
                OR toLower(coalesce(b.area, '')) CONTAINS toLower(clue)
                OR toLower(coalesce(b.branch_address, '')) CONTAINS toLower(clue)
                OR toLower(coalesce(b.city, '')) CONTAINS toLower(clue)
            )
        """)

    # Only use area as a separate filter when there are no general location clues.
    elif area:
        conditions.append("toLower(b.area) CONTAINS toLower($area)")

    if not (branch_code or city or province or area or location_clues):
        raise ValueError("Search plan does not contain any usable filters.")

    query = f"""
    MATCH (b:Branch)
    WHERE {" AND ".join(conditions)}
    RETURN properties(b) AS branch
    ORDER BY b.branch_code
    LIMIT $limit
    """

    rows = graph.query(query, params=params)
    results = [row["branch"] for row in rows]

    print(f"Local search returned {len(results)} branches.")

    return results


def global_search(plan):

    filters = plan.get("filters", {})
    city = filters.get("city") or plan.get("city")
    province = filters.get("province") or plan.get("province")

    conditions = ["b.dataset_id = $dataset_id"]

    if city:
        conditions.append("toLower(b.city) = toLower($city)")

    if province:
        conditions.append("toLower(b.province) = toLower($province)")

    query = f"""
    MATCH (b:Branch)
    WHERE {" AND ".join(conditions)}

    RETURN properties(b) AS branch
    ORDER BY b.branch_code
    LIMIT $limit
    """

    params = {
        "dataset_id": DATASET_ID,
        "city": city,
        "province": province,
        "limit": plan.get("limit") or 300,
    }

    rows = graph.query(query, params=params)

    results = [row["branch"] for row in rows]

    format_results(results, plan.get("requested_fields"))

    return results


# 3. NEAREST SEARCH

def nearest_search(plan):

    # Get coordinates automatically from the user's location
    location = geocode_location(plan)

    latitude = location["latitude"]
    longitude = location["longitude"]

    filters = plan.get("filters", {})
    city = filters.get("city") or plan.get("city")
    province = filters.get("province") or plan.get("province")

    conditions = [
        "b.dataset_id = $dataset_id",
        "b.location IS NOT NULL"
    ]

    if city:
        conditions.append("toLower(b.city) = toLower($city)")

    if province:
        conditions.append("toLower(b.province) = toLower($province)")

    query = f"""
    MATCH (b:Branch)
    WHERE {" AND ".join(conditions)}

    WITH b,
         point.distance(
             b.location,
             point({{
                 latitude: $latitude,
                 longitude: $longitude
             }})
         ) AS distance_m

    RETURN
        properties(b) AS branch,
        round(distance_m / 1000.0, 2) AS distance_km

    ORDER BY distance_m
    LIMIT $limit
    """

    rows = graph.query(query, params={
        "dataset_id": DATASET_ID,
        "latitude": latitude,
        "longitude": longitude,
        "city": city,
        "province": province,
        "limit": plan.get("limit") or 10
    })

    results = []

    for row in rows:
        branch = row["branch"]
        branch["distance_km"] = row["distance_km"]
        results.append(branch)

    print("\nGeocoded location:", location["display_name"])
    print("Coordinates:", latitude, longitude)

    format_results(
        results,
        plan.get("requested_fields", []) + ["distance_km"]
    )

    return results


# 4. ROUTER
# Selects the appropriate search function using the LLM plan.

def execute_search(plan):

    scope = plan.get("search_scope")

    print("Search scope:", scope)

    if scope == "local":
        return local_search(plan)

    elif scope == "global":
        return global_search(plan)

    elif scope == "nearest":
        return nearest_search(plan)

    else:
        raise ValueError(f"Unsupported search scope: {scope}")


In [31]:
user_question = "Tell me details about the branch with branch code 3601"

query_plan = plan_query(user_question, schema)

results = execute_search(query_plan)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Search scope: local


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


,branch_code,branch_name,branch_address,city,area
0,3601,Hub Chowki Lasbela,"Plot Bearing No. 221/222, Mouza Berot, Tehsil ...",Hub,SITE


In [34]:
user_question = "Tell me all branches in Lahore"

query_plan = plan_query(user_question, schema)

results = execute_search(query_plan)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Search scope: global


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


,branch_code,branch_name,branch_address,city,area
0,0201,"Gulberg, Lahore","60 Main Boulevard, Gulberg, Lahore",Lahore,Gulberg
1,0202,"Circular Road, Lahore","31/R/141, Circulat Road, Outside Shah Alam Gat...",Lahore,City Lahore
2,0203,"Azam Cloth Market, Lahore","19-Bismillah Block, Riaz Arts, Azam Cloth Mark...",Lahore,City Lahore
3,0204,"New Garden Town, Lahore","Commercial Flat # A, Block # 5, Ground Floor, ...",Lahore,NGT
4,0205,"DHA Phase-III (Y Block), Lahore","Plot # 97-Y, Phase III Commercial (Opposite Sh...",Lahore,Defence Lahore
...,...,...,...,...,...
137,1173,DHA Rahbar,"Plot # 3, sector 2P, Phase 11P DHA Rahbar-Lahore",Lahore,Canal
138,1174,Salamat Pura Branch,"Khewat # 94, Khasra # 177-14 Marla situated at...",Lahore,Business Lahore
139,1175,Australia Chowk Lahore,"39 Main Railway Road, Lahore",Lahore,City Lahore
140,2676,Phool Nagar,"Khewat No.895, Near PSO Petrol Pump, Phool Nag...",Lahore,Kasur


In [47]:
user_question = "Tell me the nearest Meezan branches to Gulberg III, Lahore"

query_plan = plan_query(user_question, schema)

results = execute_search(query_plan)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Search scope: nearest
Geocoding: Gulberg III, Lahore
Location found: 60/C - 1, off M.M. Alam Road, (Near Telephone Exchange), Gulberg III, Lahore, Pakistan, Gulberg III, گلبرگ, لاہور کینٹ, تحصیل لاہور کنٹونمنٹ, ضلع لاہور, لاہور ڈویژن, پنجاب, 54660, پاکستان

Geocoded location: 60/C - 1, off M.M. Alam Road, (Near Telephone Exchange), Gulberg III, Lahore, Pakistan, Gulberg III, گلبرگ, لاہور کینٹ, تحصیل لاہور کنٹونمنٹ, ضلع لاہور, لاہور ڈویژن, پنجاب, 54660, پاکستان
Coordinates: 31.5168087 74.3490447


,branch_code,branch_name,branch_address,city,area,distance_km
0,0290,"Liberty Market Branch, Lahore","Shop No. 14-17, Liberty Shopping Centre, Gulbe...",Lahore,Gulberg,0.00
1,1138,"Falcon Complex, Gulberg-III Lahore","Emporium No.3, PAF Falcon Complex, Gulberg-III...",Lahore,Gulberg,0.23
2,0245,"Ghalib Market, Lahore","64 A-II, Gulberg III, Ghalib Market Lahore.",Lahore,Gulberg,0.66
3,0254,"Zahoor Elahi Road, Lahore (Ex: HSBC)","Building No. 3-A, Block A, Gulberg II, Zahoor ...",Lahore,Gulberg,0.81
4,1160,"Tricon Corporate Center, Lahore",Office # 1201 & 1202 – Tricon Corporate Centre...,Lahore,Gulberg,0.84
5,0266,"Mini Market Gulberg II Branch, Lahore","Plot No 25-L, College Road, Gulberg II, Lahore.",Lahore,Gulberg,0.84
6,0201,"Gulberg, Lahore","60 Main Boulevard, Gulberg, Lahore",Lahore,Gulberg,1.25
7,0226,"Punjab Coopeartive Housing Society (PCHS), Lahore","66-F, Phase I, Punjab Co-operative Housing Soc...",Lahore,Defence Lahore,1.91
8,0294,"Jail Road Branch, Lahore\n(Ex: Zafar Ali Road ...","Jail Road Branch, High Q Tower, Jail Road Lahore",Lahore,Gulberg,2.10
9,0216,"Cavalry Ground, Lahore","Plot # 72, Commercial Area, Cavalry Ground Lah...",Lahore,Gulberg,2.14


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [58]:
user_question = "Tell me the Meezan branches in Gulberg, karachi"

query_plan = plan_query(user_question, schema)

results = execute_search(query_plan)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Search scope: local
No matching branches found.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [43]:
user_question = "Tell me the nearest Meezan branches to Jahangir road, lahore"

query_plan = plan_query(user_question, schema)

results = execute_search(query_plan)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Search scope: nearest
Geocoding: Jahangir Road, Lahore
Location found: Jahangir Road, ہربنس پُورہ, کوٹلی غازی, تحصیل شالیمار, ضلع لاہور, لاہور ڈویژن, پنجاب, 54850, پاکستان

Geocoded location: Jahangir Road, ہربنس پُورہ, کوٹلی غازی, تحصیل شالیمار, ضلع لاہور, لاہور ڈویژن, پنجاب, 54850, پاکستان
Coordinates: 31.5714965 74.4238846


,branch_code,branch_name,branch_address,city,area,distance_km
0,0235,"Canal Bank Road Mughalpura, Lahore","Plot # 125, St # 33, Naya Pul, Punj Pir Road, ...",Lahore,Lahore Cantt,1.28
1,0241,G.T Road Daroghawala,"Plot No.329-F, Main G.T Road, Daroghawala Lahore.",Lahore,Business Lahore,2.52
2,0244,"Mustafaabad Branch, Lahore","Meezan Bank Limited, Property # SE-6-R-148 All...",Lahore,Gulberg,2.84
3,1150,Harbanspura,"Property No.116, Saeed Block, Canal Bank, Sala...",Lahore,Lahore Cantt,3.10
4,1146,Askari-X Lahore Cantt,"Shop No.6,7,14-15, Sector F, Askari X, Lahore ...",Lahore,Lahore Cantt,4.21


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### Complete pipeline

In [ ]:

# import json

# def run_query(user_question):

#     query_plan = None
#     retrieved_facts = []

#     try:
#         # STEP 1: Generate the search plan
#         print("\nSTEP 1: Generating query plan...")

#         query_plan = plan_query(user_question, schema)

#         print(json.dumps(query_plan, indent=2, ensure_ascii=False))

#         # STEP 2: Execute the appropriate search
#         print("\nSTEP 2: Executing search...")

#         retrieved_facts = execute_search(query_plan)

#         print(f"\nRetrieved {len(retrieved_facts)} records.")

#         # STEP 3: Prepare evidence for the LLM
#         facts_json = json.dumps(
#             retrieved_facts,
#             indent=2,
#             ensure_ascii=False,
#             default=str
#         )

#         # STEP 4: Generate the final answer
#         print("\nSTEP 3: Generating final response...")

#         prompt = f"""
# You are a Meezan Bank branch information assistant.

# Answer the user's question using ONLY the retrieved database facts below.

# USER QUESTION:
# {user_question}

# RETRIEVED DATABASE FACTS:
# {facts_json}

# INSTRUCTIONS:
# 1. Answer directly and clearly.
# 2. Use only information present in the retrieved facts.
# 3. Do not invent branch names, addresses, codes, or distances.
# 4. If no facts were retrieved, clearly say that no matching branches
#    were found in the database.
# 5. If multiple branches were retrieved, present them in a readable list.
# 6. Include branch code, name, address, city, and area when available.
# 7. Include distance when it is present in the retrieved facts.
# 8. Do not claim that a branch is the nearest unless the facts include
#    distance results from a nearest search.
# 9. If the results do not fully answer the question, explain the limitation.

# FINAL ANSWER:
# """

#         llm_response = llm.invoke(prompt)

#         if hasattr(llm_response, "content"):
#             final_answer = llm_response.content
#         else:
#             final_answer = str(llm_response)

#         return final_answer, query_plan, retrieved_facts

#     except Exception as e:
#         error_message = f"Pipeline error: {str(e)}"

#         return (
#             error_message,
#             query_plan or {},
#             retrieved_facts or []
#         )


In [73]:
# updated
import json

# Coordinate fields excluded from the final LLM context for global searches.
COORDINATE_FIELDS = {
    "latitude",
    "longitude",
    "lat",
    "lon",
    "lng",
    "coordinates",
    "location_point",
    "point"
}


def run_query(user_question):

    query_plan = None
    retrieved_facts = []

    try:
        # STEP 1: Generate the search plan
        print("\nSTEP 1: Generating query plan...")

        query_plan = plan_query(user_question, schema)

        print(json.dumps(
            query_plan,
            indent=2,
            ensure_ascii=False,
            default=str
        ))

        # STEP 2: Execute the appropriate search
        print("\nSTEP 2: Executing search...")

        retrieved_facts = execute_search(query_plan)

        print(f"\nRetrieved {len(retrieved_facts)} records.")

        # STEP 3: Identify search type
        search_type = (
            query_plan.get("search_type")
            or query_plan.get("search_mode")
            or query_plan.get("strategy")
            or ""
        ).lower()

        print(f"Search type: {search_type}")

        # STEP 4: Prepare separate LLM context
        # Original retrieved_facts remain unchanged for the table.
        if search_type == "global":

            llm_facts = [
                {
                    key: value
                    for key, value in fact.items()
                    if key.lower() not in COORDINATE_FIELDS
                }
                for fact in retrieved_facts
            ]

            print("Global search: coordinates excluded from LLM context.")

        else:
            llm_facts = retrieved_facts

        facts_json = json.dumps(
            llm_facts,
            indent=2,
            ensure_ascii=False,
            default=str
        )

        # STEP 5: Generate the final answer
        print("\nSTEP 3: Generating final response...")

        prompt = f"""
You are a Meezan Bank branch information assistant.

Answer the user's question using ONLY the retrieved database facts below.

USER QUESTION:
{user_question}

RETRIEVED DATABASE FACTS:
{facts_json}

INSTRUCTIONS:
1. Answer directly and clearly.
2. Use only information present in the retrieved facts.
3. Do not invent branch names, addresses, codes, or distances.
4. If no facts were retrieved, clearly say that no matching branches
   were found in the database.
5. Do not repeat the entire retrieved dataset in your answer.
   The application will display the complete records separately.
6. Include only the most relevant branch details in your explanation.
7. Include distance only when it is present in the retrieved facts.
8. Do not claim that a branch is the nearest unless the facts include
   distance results from a nearest search.
9. If the results do not fully answer the question, explain the limitation.

FINAL ANSWER:
"""

        llm_response = llm.invoke(prompt)

        if hasattr(llm_response, "content"):
            final_answer = llm_response.content
        else:
            final_answer = str(llm_response)

        # Return original records, including coordinates.
        # These can be displayed directly in the results table.
        return final_answer, query_plan, retrieved_facts

    except Exception as e:
        error_message = f"Pipeline error: {str(e)}"

        return (
            error_message,
            query_plan or {},
            retrieved_facts or []
        )


In [62]:

user_question = "Tell me the Meezan branches in Gulberg, Karachi"

final_answer, planner_output, retrieved_facts = run_query(user_question)

print("\nFINAL ANSWER:")
print(final_answer)

print("\nPLANNER OUTPUT:")
print(json.dumps(planner_output, indent=2, ensure_ascii=False))

print("\nRETRIEVED FACTS:")
print(json.dumps(
    retrieved_facts,
    indent=2,
    ensure_ascii=False,
    default=str
))



STEP 1: Generating query plan...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{
  "intent": "branch_search",
  "search_scope": "local",
  "location_clues": [
    "Gulberg",
    "Karachi"
  ],
  "location_query": "Gulberg, Karachi",
  "location_search_queries": [
    "Gulberg Karachi",
    "Gulberg area, Karachi"
  ],
  "fallback_location_queries": [],
  "city": "Karachi",
  "province": null,
  "needs_geocoding": false,
  "needs_nearest": false,
  "branch_code": null,
  "filters": {
    "city": "Karachi",
    "province": null,
    "area": "Gulberg"
  },
  "requested_fields": [
    "branch_code",
    "branch_name",
    "branch_address",
    "city",
    "area"
  ],
  "ambiguities": []
}

STEP 2: Executing search...
Search scope: local
No matching branches found.

Retrieved 0 records.

STEP 3: Generating final response...

FINAL ANSWER:
No matching Meezan Bank branches in Gulberg, Karachi were found in the database.

PLANNER OUTPUT:
{
  "intent": "branch_search",
  "search_scope": "local",
  "location_clues": [
    "Gulberg",
    "Karachi"
  ],
  "location_query": "

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [75]:

import gradio as gr
import pandas as pd


def gradio_run(user_question):

    if not user_question or not user_question.strip():
        return (
            "Please enter a question.",
            {},
            pd.DataFrame(),
            []
        )

    final_answer, planner_output, retrieved_facts = run_query(
        user_question.strip()
    )

    # Convert retrieved Neo4j records into a table.
    if retrieved_facts:
        facts_df = pd.DataFrame(retrieved_facts)
    else:
        facts_df = pd.DataFrame()

    # Return both the table and original JSON records.
    return (
        final_answer,
        planner_output,
        facts_df,
        retrieved_facts
    )


with gr.Blocks(
    title="Meezan Branch Search",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("# Meezan Bank Branch Search")
    gr.Markdown(
        "Ask about branches, locations, branch codes, or nearest branches."
    )

    user_input = gr.Textbox(
        label="Your Question",
        placeholder="e.g. Tell me the Meezan branches in Gulberg, Karachi",
        lines=2
    )

    search_button = gr.Button(
        "Search Branches",
        variant="primary"
    )

    gr.Markdown("## Final LLM Response")
    final_output = gr.Markdown()

    gr.Markdown("## Retrieved Branch Records")

    facts_table = gr.Dataframe(
        label="Neo4j Retrieved Results",
        interactive=False,
        wrap=True
    )

    with gr.Accordion("Planner Output (JSON)", open=False):
        planner_output = gr.JSON(label="Query Plan")

    with gr.Accordion("Raw Retrieved Database Facts (JSON)", open=False):
        facts_output = gr.JSON(label="Complete Neo4j Results")

    search_button.click(
        fn=gradio_run,
        inputs=user_input,
        outputs=[
            final_output,
            planner_output,
            facts_table,
            facts_output
        ]
    )

    user_input.submit(
        fn=gradio_run,
        inputs=user_input,
        outputs=[
            final_output,
            planner_output,
            facts_table,
            facts_output
        ]
    )


demo.launch(
    share=False,
    debug=True
)


/tmp/ipykernel_16902/2060989522.py:34: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.



STEP 1: Generating query plan...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{
  "intent": "branch_search",
  "search_scope": "nearest",
  "location_clues": [
    "shahrahe Quaid-e-Azam, Allama Iqbal Campus, Lahore"
  ],
  "location_query": "shahrahe Quaid-e-Azam, Allama Iqbal Campus, Lahore",
  "location_search_queries": [
    "Allama Iqbal Campus, Lahore",
    "Quaid-e-Azam Road, Lahore"
  ],
  "fallback_location_queries": [
    "Allama Iqbal Campus, Lahore",
    "Quaid-e-Azam Road, Lahore"
  ],
  "city": "Lahore",
  "province": null,
  "needs_geocoding": true,
  "needs_nearest": true,
  "branch_code": null,
  "filters": {
    "city": "Lahore",
    "province": null,
    "area": null
  },
  "requested_fields": [
    "branch_code",
    "branch_name",
    "branch_address",
    "city",
    "area"
  ],
  "ambiguities": []
}

STEP 2: Executing search...
Search scope: nearest
Geocoding: shahrahe Quaid-e-Azam, Allama Iqbal Campus, Lahore
Geocoding: Allama Iqbal Campus, Lahore
Geocoding: Quaid-e-Azam Road, Lahore


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


... (9 outputs left out)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{
  "intent": "branch_search",
  "search_scope": "nearest",
  "location_clues": [
    "pucit lahore"
  ],
  "location_query": "pucit lahore",
  "location_search_queries": [
    "pucit, lahore",
    "Punjab University College of Information Technology, Lahore"
  ],
  "fallback_location_queries": [
    "Nearby roads or localities around PUCIT, Lahore",
    "University Road, Lahore"
  ],
  "city": "Lahore",
  "province": null,
  "needs_geocoding": true,
  "needs_nearest": true,
  "branch_code": null,
  "filters": {
    "city": "Lahore",
    "province": null,
    "area": null
  },
  "requested_fields": [
    "branch_code",
    "branch_name",
    "branch_address",
    "city",
    "area"
  ],
  "ambiguities": []
}

STEP 2: Executing search...
Search scope: nearest
Geocoding: pucit lahore
Geocoding: pucit, lahore
Geocoding: Punjab University College of Information Technology, Lahore


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Keyboard interruption in main thread... closing server.
